## Model Evaluation: Walk-Forward Cross Validation

Before comparing SARIMA forecasts against live World Cup data, the modeel's accuracy will be validated using walk-forward cross-validation. This ensures the baseline forecasts are reliable enough to meaningfully measure World Cup deviations.

**Methodology:**
- Split each city's time series into 4 folds
- Each fold trains on all data before the test window (no data leakage)
- Test window is 6 months per fold
- Metrics: RMSE and MAPE averaged across all folds and cities

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import TimeSeriesSplit 
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima


In [5]:
# Reading in data
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date', parse_dates = True)
print(df.head())

            spend_all  spend_acf     cityname stateabbrev  city_pop2019  \
Date                                                                      
2022-01-31    -0.1300    -0.1580  Los Angeles          CA      10039107   
2022-01-31     0.1810     0.0257      Chicago          IL       5150233   
2022-01-31    -0.0201    -0.0351       Dallas          TX       2635516   
2022-01-31     0.0388     0.0112       Austin          TX       1273954   
2022-01-31    -0.0339    -0.0717    Charlotte          NC       1110356   

            season host_status  
Date                            
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter    Non-Host  


In [6]:
# Testing on Atlanta data to ensure pipeline works end-to-end and for debugging purposes

atl_df = df[df['cityname'] == 'Atlanta']
atl_spending = atl_df.spend_acf

# Splitting data into 3 folds

time_split = TimeSeriesSplit(n_splits=2, test_size=12)  # 2 splits with a test size of 12 months each

rmse_scores = [] # store rmse scores to calculate average at the end of loop
mape_scores = [] # store mape scores to calculate average at the end of loop

# Loop through the splits and train/test the model
for train_idx, test_idx in time_split.split(atl_spending):

    atl_train, atl_test = atl_spending.iloc[train_idx], atl_spending.iloc[test_idx]
    
    # Fitting SARIMAX model
    auto_model = auto_arima(atl_train, seasonal=True, m=12, D=1, max_D=1, error_action='ignore', suppress_warnings=True)
    sarima_model = SARIMAX(atl_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
    model_fit = sarima_model.fit(disp=False) 
    
    # Generate forecasts from training data
    forecast = model_fit.get_forecast(steps=len(atl_test))
    forecast_mean = forecast.predicted_mean
    forecast_mean.name = "Predicted Forecast"  

    # Computing error Calculation, RMSE (Root Mean Squared Error)
    mse = mean_squared_error(atl_test, forecast_mean)
    rmse = np.sqrt(mse) 
    print(f"RMSE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {rmse:.4f}")
    rmse_scores.append(rmse)

    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((atl_test.values - forecast_mean.values) / atl_test.values)) * 100
    print(f"MAPE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {mape:.4f}")
    mape_scores.append(mape)
    print()

RMSE (Jun 2024 - May 2025): 0.0629
MAPE (Jun 2024 - May 2025): 36.4004

RMSE (Jun 2025 - May 2026): 0.0424
MAPE (Jun 2025 - May 2026): 29.5924



RMSE is used over MSE because it returns error in the original units of the data (percent change), making it directly interpretable. A lower RMSE indicates the model's forecasts are closer to actual spending values.